# 14. Built-in SQL Functions

*Module: 4 — Advanced querying · Companion summary: `lectures/14-builtin-sql-functions.md` · SQL engine: SQLite (in-memory)*

## What you will study

- aggregate functions: SUM, MIN, MAX, AVG, COUNT
- scalar (string) functions: ROUND, LENGTH, UPPER, LOWER
- combine them: scalar inside aggregate, functions in WHERE and SELECT

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

In [ ]:
import sqlite3
import textwrap

conn = sqlite3.connect(':memory:')  # throw-away in-memory database
conn.execute("PRAGMA foreign_keys = ON")

def q(sql):
    """Run a query, print formatted SQL, and display the result set as a table."""
    clean_sql = textwrap.dedent(sql).strip()

    # 1. Print formatted SQL query
    print("\n" + "=" * 60)
    print("QUERY:")
    print(textwrap.indent(clean_sql, "  "))
    print("-" * 60)

    cur = conn.execute(sql)
    rows = cur.fetchall()

    if cur.description:  # SELECT / statement returning rows
        headers = [d[0] for d in cur.description]
        str_rows = [[("" if v is None else str(v)) for v in r] for r in rows]

        # Calculate dynamic column widths for clean alignment
        widths = [
            max(len(h), *(len(r[i]) for r in str_rows), 1)
            for i, h in enumerate(headers)
        ] if str_rows else [len(h) for h in headers]

        header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
        sep_line = "-+-".join("-" * widths[i] for i in range(len(headers)))

        print(header_line)
        print(sep_line)
        for r in str_rows:
            print(" | ".join(val.ljust(widths[i]) for i, val in enumerate(r)))
        if not str_rows:
            print("(0 rows returned)")
    else:  # DDL / DML (CREATE, INSERT, UPDATE, DELETE)
        affected = cur.rowcount
        print(f"Done (rows affected: {affected})" if affected != -1 else "Done.")

    print("=" * 60)

print('Connection ready:', conn)

## 🧪 Sample data — PETRESCUE (animal rescue records)

Columns: rescue id, animal, quantity rescued, cost of rescue, date.

In [ ]:
q("CREATE TABLE petrescue (id INTEGER PRIMARY KEY, animal TEXT, quantity INTEGER, cost REAL, rescue_date TEXT)")
q("""
INSERT INTO petrescue VALUES
    (1, 'Dog',   1, 100.0, '2018-05-01'),
    (2, 'Cat',   1,  50.0, '2018-05-10'),
    (3, 'Dog',   2, 120.0, '2018-06-15'),
    (4, 'Bird',  3,  60.0, '2019-05-02'),
    (5, 'Cat',   2,  80.0, '2019-06-20'),
    (6, 'Dog',   1, 150.0, '2020-01-05'),
    (7, 'Bird',  5,  75.0, '2020-02-14'),
    (8, 'Cat',   1,  45.0, '2020-05-30')
""")
q("SELECT * FROM petrescue")

## 🧪 SUM — and rename the output with AS

In [ ]:
q("SELECT SUM(cost) AS total_cost FROM petrescue")

## 🧪 MIN / MAX — with and without WHERE

In [ ]:
q("SELECT MIN(cost) FROM petrescue")
q("SELECT MIN(cost) FROM petrescue WHERE animal = 'Dog'")
q("SELECT MAX(cost) FROM petrescue")
q("SELECT MAX(cost) FROM petrescue WHERE animal = 'Dog'")

## 🧪 AVG — and nesting a scalar function inside it

In [ ]:
q("SELECT AVG(cost) FROM petrescue")
q("SELECT ROUND(AVG(cost), 2) AS avg_cost FROM petrescue WHERE animal = 'Dog'")
q("SELECT ROUND(AVG(cost / quantity), 2) AS avg_cost_per_animal FROM petrescue WHERE animal = 'Dog'")

## 🧪 Scalar functions change values row by row

In [ ]:
q("SELECT animal, UPPER(animal) AS up, LOWER(animal) AS low, LENGTH(animal) AS len FROM petrescue")

## 🧪 Scalar functions as WHERE predicates

In [ ]:
q("SELECT id, animal, cost FROM petrescue WHERE LOWER(animal) = 'dog'")
q("SELECT id, animal FROM petrescue WHERE LENGTH(animal) = 4")

## 🧪 Nesting: DISTINCT(UPPER(...)) — dedupe after transforming

In [ ]:
q("SELECT DISTINCT(UPPER(animal)) AS animal_type FROM petrescue")

## 💡 Tips & tricks

- **Aggregate** functions collapse many rows → one value; **scalar** functions map
  one row → one value. Saying the distinction out loud fixes it in memory.
- ROUND(x, n) exists in SQLite and DB2; check your engine's list before assuming.
- 📕 Dialect note: DB2 has `TRUNCATE`-like `TRUNC`, Oracle `RPAD/LPAD` etc.; SQLite's
  built-ins are leaner — most string manipulation is often better done in pandas.

## 🎯 Your turn

1. Total cost of rescuing birds.
2. The average quantity per rescue, rounded to 1 decimal.
3. Distinct animals in lowercase.

In [ ]:
# 💡 Solution
q("SELECT SUM(cost) FROM petrescue WHERE animal = 'Bird'")
q("SELECT ROUND(AVG(quantity), 1) FROM petrescue")
q("SELECT DISTINCT(LOWER(animal)) FROM petrescue")